# Loan Eligibility Prediction, step by step

This notebook uses the CSV already in your project. You will learn how to inspect a dataset, distinguish features from the target, visualize missing values and relationships, train several classifiers, and judge their predictions on unseen data.

**Run the cells from top to bottom.** Each code cell depends on earlier cells. In Jupyter choose **Kernel > Restart Kernel and Run All Cells** to check that the whole notebook works. Pause at each explanation and practice question.

We predict the **recorded approval decision**, not repayment or objective creditworthiness. This is a small educational dataset with unverified provenance and units. We keep the raw CSV unchanged.

## 1. Imports and paths

A **library** provides reusable tools. pandas handles tables; NumPy handles numerical arrays; Matplotlib makes charts; scikit-learn supplies preprocessing, splitting, models and evaluation. `Path` builds file paths without depending on a Windows-specific separator.

The path search below supports launching Jupyter from the repository root, project folder or notebook folder. If no input file is found it raises a useful error rather than silently loading different data.

In [ ]:
from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

project_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "projects" / "03_loan_eligibility",
]
PROJECT_DIR = next(
    (p.resolve() for p in project_candidates
     if (p / "data" / "raw" / "loan_prediction.csv").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open Jupyter from the repository, project, or notebook folder.")
DATA_PATH = PROJECT_DIR / "data" / "raw" / "loan_prediction.csv"
SEED = 42
plt.style.use("seaborn-v0_8-whitegrid")
print("Using:", DATA_PATH)

## 2. Load the data and inspect its structure

`pd.read_csv()` reads the file into a **DataFrame**, a table with named columns.
`head()` previews the first five rows. `shape` is a pair: (number of rows, number of columns).
`info()` prints data types and nonmissing counts; do not wrap it in `print()`, since it already prints and returns `None`.

A row is an application; a column is an attribute. The expected structure is **614 rows and 13 columns**. pandas may report text as `str`, `string` or `object`, depending on version.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Rows and columns:", df.shape)
display(df.head())
df.info()

In [ ]:
display(df.describe().T)
display(df.describe(include=["str", "object"]).T)

For numeric columns, **count** excludes missing values; **mean** is the arithmetic average; **std** is standard deviation; **min/max** are extremes; **50%** is the median. The 25% and 75% values are quartiles. For categories, **unique** counts distinct nonmissing values, **top** is the most common value and **freq** is its count.

Applicant income has a median of **3,812.5**, mean of about **5,403.46**, and maximum of **81,000**. The mean being higher than the median suggests a right tail; the chart below lets us inspect that.

No metadata confirms currency, income period, amount scale or term unit. We therefore describe values in supplied units and avoid inventing an affordability ratio.

## 3. Features and target

The **target** is the answer we want the model to predict: `Loan_Status`.
The other useful inputs are **features**. This is **supervised binary classification**:
supervised because example answers are available, binary because there are two classes.

| Field | Meaning / observed values | Treatment |
| --- | --- | --- |
| Loan_ID | Application identifier | Exclude |
| Gender | Male / Female | Categorical |
| Married | Yes / No | Categorical |
| Dependents | 0, 1, 2, 3+ | Categorical |
| Education | Graduate / Not Graduate | Categorical |
| Self_Employed | Yes / No | Categorical |
| ApplicantIncome | Applicant income, units unconfirmed | Numeric |
| CoapplicantIncome | Coapplicant income, units unconfirmed | Numeric |
| LoanAmount | Loan amount, units unconfirmed | Numeric |
| Loan_Amount_Term | Term, time unit unconfirmed | Numeric |
| Credit_History | 0 / 1, original definition unconfirmed | Binary |
| Property_Area | Rural / Semiurban / Urban | Categorical |
| Loan_Status | Y = Approved, N = Not Approved | Target |

We exclude the ID because it is a record label. Including `Loan_Status` in the inputs would reveal the answer: **target leakage**. Treating `3+` dependents as a category avoids pretending it is an exact count.

In [ ]:
NUMERIC_FEATURES = [
    "ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"
]
CATEGORICAL_FEATURES = [
    "Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"
]
BINARY_FEATURES = ["Credit_History"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
LABEL_NAMES = ["Not Approved", "Approved"]

for column in CATEGORICAL_FEATURES + BINARY_FEATURES + ["Loan_Status"]:
    print(column, ":", df[column].dropna().unique().tolist())

if df["Loan_Status"].isna().any() or set(df["Loan_Status"].unique()) != {"Y", "N"}:
    raise ValueError("Every training row must have a known Y or N target.")
target_summary = df["Loan_Status"].value_counts().reindex(["N", "Y"]).to_frame("count")
target_summary["percent"] = 100 * target_summary["count"] / len(df)
display(target_summary)

There are **422 Approved (68.73%)** and **192 Not Approved (31.27%)** applications.

A classifier that always predicts Approved would score 68.73% on the full data while identifying **none** of the Not Approved applications. This is **class imbalance**. Accuracy alone can hide it. Later we train a majority baseline and report recall for both classes.

**Check your understanding:** Is predicting loan amount the same kind of problem? No: an amount is a numerical target, so that would be regression. Logistic regression, despite its name, is used here for classification.

## 4. Data quality: missing values and duplicates

`isna()` creates a True/False table of missing cells. Summing counts True as 1.
A missing value means “not recorded,” which is different from zero. For example, zero coapplicant income is a recorded value.

An exact duplicate repeats every field; a repeated ID could represent the same application and needs investigation. We check both. Repeated applications in training and test data could make evaluation misleading.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)
display(missing)
print("Missing cells:", int(df.isna().sum().sum()))
print("Rows with at least one missing cell:", int(df.isna().any(axis=1).sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Repeated IDs:", int(df["Loan_ID"].duplicated().sum()))
print("Missing targets:", int(df["Loan_Status"].isna().sum()))

There are **149 missing cells**, most in `Credit_History` (50), then `Self_Employed` (32). There are no duplicate rows or repeated IDs in this file, and the target is complete.

We retain incomplete rows and **impute** (fill) missing features using training data. Dropping every incomplete row would discard useful examples. We never fill unknown target labels by guessing. The reusable script also validates the expected columns, IDs, numeric values and binary credit codes.

**Check:** Why can the number of missing cells exceed the number of incomplete rows? One application can have several missing fields.

## 5. Visualize distributions and relationships

Exploratory data analysis (**EDA**) helps identify patterns and questions before fitting models. These full-file charts are descriptive. We do not use them to choose features, model settings or decision thresholds. For a strict prospective experiment, set aside test data first and do target-driven EDA only on training data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df["Loan_Status"].value_counts().reindex(["N", "Y"])
axes[0].bar(LABEL_NAMES, counts, color=["#cf6660", "#27807c"])
axes[0].set(title="Recorded decisions", ylabel="Applications")
for i, count in enumerate(counts):
    axes[0].text(i, count + 5, f"{count} ({count / len(df):.1%})", ha="center")
missing.loc[missing["missing_count"] > 0, "missing_count"].sort_values().plot.barh(ax=axes[1])
axes[1].set(title="Missing values", xlabel="Applications with missing value")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, column in zip(axes.flat, NUMERIC_FEATURES):
    ax.hist(df[column].dropna(), bins=25, color="#27807c", edgecolor="white")
    ax.set(title=column, xlabel="Value in supplied units", ylabel="Applications")
plt.tight_layout()
plt.show()

Income and loan amount have long right tails: many moderate values and a few very large ones. The term field is concentrated at 360 in its unconfirmed time units. Large values may be valid; do not delete them only because they are unusual. Standardization later changes units but does **not** remove skew or outliers.

In [ ]:
# Explicitly include missing credit history rather than dropping those rows.
credit = df["Credit_History"].map({0: "0", 1: "1"}).fillna("Missing")
credit_counts = pd.crosstab(credit, df["Loan_Status"]).reindex(columns=["N", "Y"])
credit_rates = credit_counts.div(credit_counts.sum(axis=1), axis=0)
display(credit_counts)
display((100 * credit_rates).round(2).rename(columns={"N": "Not Approved %", "Y": "Approved %"}))
ax = credit_rates.rename(columns={"N": "Not Approved", "Y": "Approved"}).plot.bar(
    stacked=True, figsize=(8, 4), color=["#cf6660", "#27807c"], rot=0
)
ax.set(title="Decision proportions by recorded credit history", ylabel="Proportion", xlabel="Credit_History")
plt.tight_layout()
plt.show()

For code 1, **378 of 475 (79.58%)** are Approved; for code 0, **7 of 89 (7.87%)** are Approved. Among 50 missing credit-history records, 37 are Approved. Counts matter: percentages from small groups are less stable.

This relationship is **association**, not proof that changing a credit code causes approval. We also lack the original definition of the credit-history variable.

In [ ]:
# Explore other categories and numeric associations, with group sizes visible.
for column in CATEGORICAL_FEATURES:
    table = pd.crosstab(df[column].fillna("Missing"), df["Loan_Status"]).reindex(columns=["N", "Y"])
    table["total"] = table["N"] + table["Y"]
    table["approved_percent"] = 100 * table["Y"] / table["total"]
    print("\n" + column)
    display(table.round(2))

numeric_eda = df[NUMERIC_FEATURES + BINARY_FEATURES].copy()
numeric_eda["Approved"] = df["Loan_Status"].map({"N": 0, "Y": 1})
display(numeric_eda.corr().round(2))

The correlation table summarizes **linear association** between numeric columns; pandas uses available pairs when values are missing. Correlation is not causation, and low correlation does not rule out nonlinear patterns or interactions. Demographic approval differences describe this sample and do not establish which variables ought to influence a decision.

We keep the predeclared feature set and do not choose a model using these whole-file target relationships.

## 6. Split the data before learning transformations

We use `X` for features and `y` for encoded labels: N = 0, Y = 1. The label mapping is a fixed rule, so it does not learn anything from the test set.

An **80/20 split** leaves 491 training rows and 123 test rows. `stratify=y` keeps approximate class proportions in each part. `random_state=42` makes the random split repeatable.

The test set is an exam: keep it aside until model selection is complete. Imputation, scaling and encoding must also be learned without looking at that exam.

In [ ]:
X = df[FEATURES].copy()
y = df["Loan_Status"].map({"N": 0, "Y": 1})
if df["Loan_ID"].isna().any() or df["Loan_ID"].duplicated().any():
    raise ValueError("Investigate missing or repeated application IDs before splitting.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
display(pd.DataFrame({
    "training": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
}).rename(index={0: "Not Approved", 1: "Approved"}))
assert set(X_train.index).isdisjoint(X_test.index)

## 7. Build preprocessing inside a pipeline

A **Pipeline** executes steps in order. A **ColumnTransformer** sends different columns to different steps.

- Numeric fields: replace missing values with the **training median**, then standardize using the training mean and standard deviation. The median is less affected by extreme values than the mean. Scaling helps regularized logistic regression handle different feature units.
- Categories: replace missing values with the **most frequent training category**, then **one-hot encode** each category into 0/1 indicators. We do not give Rural, Urban and Semiurban an invented numerical order.
- Binary credit history: replace missing entries with its **most frequent training code**, retaining 0/1. This simple assumption can lose information; a missingness indicator is a reasonable later experiment.

`handle_unknown="ignore"` prevents a new category from crashing prediction. It becomes all-zero indicators for that field; the model still has not learned its meaning.

We define a function returning a fresh transformer so each model gets independent preprocessing. Tree models do not need scaling; sharing the design keeps this lesson simple.

In [ ]:
def build_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("credit", SimpleImputer(strategy="most_frequent"), BINARY_FEATURES),
    ], remainder="drop")

**Why not fill every gap now?** If we computed medians from all 614 applications, test values would influence training. This is data leakage. Keeping preprocessing inside the pipeline means each fit learns only from the data passed to it, including each training fold in cross-validation. See [scikit-learn's leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

## 8. Compare algorithms on training data

| Algorithm | Idea |
| --- | --- |
| Majority baseline | Always predicts the most common training label; learns no applicant relationships |
| Logistic regression | Learns a weighted combination of encoded features and passes it through a logistic function |
| Decision tree | Learns a sequence of splits; limiting depth and leaf size reduces overfitting |
| Random forest | Averages many randomized trees; complexity does not guarantee better generalization |

**Overfitting** means fitting training patterns too closely and performing worse on unseen examples.

We use fixed teaching settings and **five-fold stratified cross-validation** on training data only. Each model trains on four folds and validates on the fifth, repeated five times. Every fold learns its own imputation, scaling and encoding.

Before seeing test scores, choose **mean balanced accuracy** for selection: it averages recall for the two classes, weighting them equally. This is a teaching choice, not a lender's economic cost function. Approved precision, recall, F1 and ROC AUC are also reported.

In [ ]:
classifiers = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Decision tree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=SEED),
    "Random forest": RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=SEED, n_jobs=1
    ),
}
candidates = {
    name: Pipeline([("preprocessor", build_preprocessor()), ("classifier", classifier)])
    for name, classifier in classifiers.items()
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc"]
rows = []
for name, candidate in candidates.items():
    fold_scores = cross_validate(
        candidate, X_train, y_train, cv=cv, scoring=scoring, error_score="raise", n_jobs=1
    )
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = fold_scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = fold_scores[f"test_{metric}"].std()
    rows.append(row)
cv_results = pd.DataFrame(rows).sort_values(
    "balanced_accuracy_mean", ascending=False, kind="stable"
).reset_index(drop=True)
display(cv_results.round(4))

The mean summarizes five validation scores. The standard deviation shows variation across folds; it is **not a confidence interval**. A small numerical lead between models is not proof of a meaningful advantage.

On the supplied data, logistic regression has the highest mean balanced accuracy, about **0.6910**. Random forest is close at about **0.6892**. We follow the declared rule rather than changing the selection metric after seeing results.

The majority baseline has Approved recall 1.0 but Not Approved recall 0.0, so its balanced accuracy is 0.5. This is why high Approved recall alone is insufficient.

## 9. Fit the selected model, then open the test set

`clone()` creates a fresh unfitted copy. `fit()` now learns preprocessing and the classifier from all training rows. `predict()` assigns labels to test applications, and `predict_proba()` provides class scores.

We define Approved = 1 as the **positive class**. “Positive” is a bookkeeping convention, not a claim of financial quality.

In [ ]:
best_name = cv_results.iloc[0]["model"]
best_model = clone(candidates[best_name])
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
p_approved = best_model.predict_proba(X_test)[:, 1]

baseline = clone(candidates["Majority baseline"]).fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_approved": precision_score(y_test, y_pred, zero_division=0),
    "recall_approved": recall_score(y_test, y_pred, zero_division=0),
    "f1_approved": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, p_approved),
}
print("Selected model:", best_name)
display(pd.Series(test_metrics, name="test score").to_frame().round(4))
print("Majority baseline test accuracy:", accuracy_score(y_test, baseline_pred))
print(classification_report(
    y_test, y_pred, labels=[0, 1], target_names=LABEL_NAMES, zero_division=0
))

| Metric | What it tells you |
| --- | --- |
| Accuracy | Fraction of all predictions that are correct |
| Approved precision | Of predicted Approved applications, how many are actually labeled Approved? |
| Approved recall | Of actually Approved applications, how many did we identify? |
| Approved F1 | Harmonic mean of Approved precision and recall |
| Balanced accuracy | Average recall across the two classes |
| ROC AUC | How well scores rank Approved above Not Approved across thresholds |

For this split, logistic regression gives **86.18% accuracy**, versus **69.11%** for the baseline. Balanced accuracy is **78.36%**, Approved precision **84.00%**, Approved recall **98.82%**, F1 **90.81%**, and ROC AUC about **0.8523**.

The 123-row test sample is small. It happens to score higher than the CV mean; that does not mean the model improved after selection. Different samples produce different estimates. We did not select using test scores.

ROC AUC is not accuracy. High scores do not establish calibrated real-world probabilities. Read the Not Approved class too: its recall is only about **57.89%**.

In [ ]:
matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])
display(pd.DataFrame(matrix, index=LABEL_NAMES, columns=LABEL_NAMES).rename_axis(
    index="Actual", columns="Predicted"
))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=[0, 1], display_labels=LABEL_NAMES,
    cmap="Blues", colorbar=False, ax=axes[0]
)
axes[0].set_title("Held-out confusion matrix")
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("Held-out ROC curve")
plt.tight_layout()
plt.show()

Read **rows as actual** and **columns as predicted**:

- **22 true negatives:** correctly predicted Not Approved.
- **16 false positives:** predicted Approved, actually labeled Not Approved.
- **1 false negative:** predicted Not Approved, actually labeled Approved.
- **84 true positives:** correctly predicted Approved.

Thus, accuracy is (22 + 84) / 123 = **86.18%**. Approved precision is 84 / (84 + 16) = **84%**. Approved recall is 84 / 85 = **98.82%**. Not Approved recall is 22 / 38 = **57.89%**. Balanced accuracy is the average of the two recalls, **78.36%**.

These errors are against historical approval labels; they are not defaults or repayment failures. The ROC curve plots true positive rate against false positive rate as the score threshold varies.

**Practice:** What happens if a model always predicts Approved? The whole matrix moves into the Approved column; no Not Approved cases are detected.

## 10. Predict a fictional new application

Provide a one-row DataFrame with the **same raw feature names**. Do not include a target or manually encode categories. The fitted pipeline repeats exactly the transformations it learned from training data.

These numbers are illustrative in the dataset's unconfirmed units. A prediction is an example of using the model, not a lending recommendation.

In [ ]:
new_application = pd.DataFrame([{
    "Gender": "Female", "Married": "Yes", "Dependents": "1",
    "Education": "Graduate", "Self_Employed": "No",
    "ApplicantIncome": 5000, "CoapplicantIncome": 1500,
    "LoanAmount": 120, "Loan_Amount_Term": 360,
    "Credit_History": 1, "Property_Area": "Urban",
}])
new_label = int(best_model.predict(new_application)[0])
new_score = float(best_model.predict_proba(new_application)[0, 1])
print("Predicted recorded status:", LABEL_NAMES[new_label])
print(f"Model score for Approved: {new_score:.3f}")

## 11. Save and reload the complete pipeline

Save the preprocessing and classifier **together** so a future prediction uses the same medians, scales and category columns. The saved model remains fitted on training rows only. Record library versions and use matching versions when reloading.

Only load trusted joblib files: deserialization can execute code. Here we reload the artifact we just created.

In [ ]:
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "loan_eligibility_pipeline.joblib"
joblib.dump(best_model, model_path)
reloaded_model = joblib.load(model_path)
np.testing.assert_array_equal(reloaded_model.predict(X_test), y_pred)
print("Saved and verified:", model_path)

# src/train.py also exports full exploration tables, CV results, predictions,
# a split manifest, dataset fingerprint, software versions, and figures.

## 12. What have you learned?

You loaded a CSV, inspected types and distributions, distinguished inputs from labels, checked missingness and duplicates, explored associations, split the data, built preprocessing that avoids leakage, compared a baseline with three algorithms, and evaluated on unseen rows.

The model beats the majority baseline on this split, but it still misses many Not Approved labels. This dataset is small; source, units, sampling and dates are unknown. Historical decisions can contain bias, and demographic fields or proxies can reproduce it. Removing a single field cannot by itself establish fairness. Approval status is not a repayment outcome. This exercise does not establish suitability for actual lending.

**Next exercises:**

1. Explain why the baseline has high Approved recall but balanced accuracy of 0.5.
2. Explain why imputation must be fitted again inside every CV fold.
3. Describe the difference between a missing numeric value and a recorded zero.
4. Explain why `Dependents="3+"` remains categorical.
5. As a separate training-only experiment, compare log-transformed incomes or missingness indicators using CV. Do not select improvements by repeatedly checking this test set; obtain a fresh final evaluation set after repeated revisions.
6. Consider why a predicted score is not automatically a calibrated probability or an objective decision.

Further reading: [pandas CSV loading](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [scikit-learn leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html), [cross-validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html).
The project [README](../README.md) and generated [findings](../reports/findings.md) provide a companion reference.